# 01a_T8_acquire_standalone

Uses [LJM library in Python](https://github.com/labjack/labjack-ljm-python) only.

This is a standalone example of acquiring data from a LabJack T8 device using the
[LJM library in Python](https://github.com/labjack/labjack-ljm-python). It demonstrates how to:
1. List connected LabJack devices.
2. Open a connection to the first available device.
3. Configure and start a streaming acquisition from specified analog input channels.
4. Collect and store the acquired data in a NumPy array.

**Note 1**: Adjust the acquisition parameters (channels, sample rate, etc.) as needed
for your specific use case and device capabilities.

**Note 2**: This example uses only the LJM library, which is the recommended way to
interface with LabJack devices in Python. There is also a lower-level library called
LJME that can be used for more direct control, but LJM provides a more user-friendly
API and is generally sufficient for most applications.

**Note 3**: No use of ophyd or bluesky in this example, as it focuses on direct
interaction with the LabJack device using the LJM library. For integration with
ophyd/bluesky, see other examples in this project.


# Preamble

## Preamble Main Libraries and a few usefull constants

In [ ]:
import sys  # noqa: F401
import os  # noqa: F401

from labjack import ljm
import numpy as np
import time
from datetime import datetime

import pandas as pd
import plotly.graph_objects as go

# sys.path.append(os.environ['USERPROFILE'] + '\\workspace\\python\\libs\\wgTools')
# import wgutils as wgu
from scipy import constants

deg2rad = np.deg2rad(1)
rad2deg = np.rad2deg(1)
eps = np.finfo(np.float64).eps
hc = constants.value("inverse meter-electron volt relationship")

## Preamble Plotly

In [ ]:
import plotly.colors

colors_plotly = plotly.colors.DEFAULT_PLOTLY_COLORS


def custom_layout(xlabel=None, ylabel=None, title=None, template="seaborn"):
    return go.Layout(
        template=template,  # Set the template
        width=800,  # Set the figure width
        height=600,  # Set the figure height
        font=dict(family="Georgia", size=14, color="#3B3B3B"),  # General font settings
        title=dict(
            text=title,
            font=dict(size=24),
            x=0.5,
            xanchor="center",
        ),
        xaxis_title=dict(text=xlabel, font=dict(size=18)),
        yaxis_title=dict(text=ylabel, font=dict(size=18)),
        xaxis=dict(tickfont=dict(size=16)),  # Set X-axis tick font size
        yaxis=dict(tickfont=dict(size=16)),  # Set Y-axis tick font size
    )


# initialize figures with something like
# plotly_fig = go.Figure(layout=custom_layout("Frequency [Hz]", "Amplitude [Volts]", fname))


## Local Functions

In [ ]:
def datenow_str():
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def timenow_str():
    return datetime.now().strftime("%H:%M:%S.%f")[:-3]


def printc(s: str, color: str = "", bold: bool = False) -> str:
    """Return the string wrapped in ANSI color/bold escape codes.

    Args:
        s: The input string to colorize.
        color: Optional color name (red, green, blue, purple, cyan).
        bold: If True, make the text bold.

    Returns:
        The ANSI-escaped string for terminal display.
    """

    color_codes = {
        "red": "91",
        "green": "92",
        "blue": "94",
        "purple": "95",
        "cyan": "96",
    }

    if color not in color_codes:
        return s

    color_code = color_codes.get(color, "")  # type: ignore
    bold_code = "1;" if bold else ""
    print(f"\033[{bold_code}{color_code}m{s}\033[0m")


def print_log(s: str):
    printc(f"[{timenow_str()}] : {s}", color="purple", bold=False)


def print_info(s: str):
    printc(f"[{timenow_str()}] : [INFO] {s}", color="blue", bold=False)


def print_warning(s: str):
    printc(f"[{timenow_str()}] : [WARNING] {s}", color="red", bold=False)


def print_error(s: str):
    printc(f"[{timenow_str()}] : [ERROR] {s}", color="red", bold=True)


def print_done(s: str):
    printc(f"[{timenow_str()}] : [DONE] {s}", color="green", bold=True)


# Setup Script Flags

In [ ]:
save_plot = False
close_connections_at_end = False

# Setup Hardware

## List Labjack Devices

In [ ]:
print_info("Searching for connected Devices...")
try:
    res = ljm.listAllS("ANY", "ANY")
    print_info(f"Devices found: {res[0]}")
    for i in range(res[0]):
        printc(
            f"\t- Device {i}: {res[1][i]}, Connection: {res[2][i]}, Serial: {res[3][i]}, IP: {res[4][i]},", color="blue"
        )
except ljm.LJMError as e:
    print_error(f"Failed to list devices: {e}")
    ljm.closeAll()

In [ ]:
print_info("Inspecting raw results from ljm.listAllS('ANY', 'ANY'):")
print(res)

## Opening connection to device with Serial number


In [ ]:
_serial_to_connection = str(480011020)  # open by serial number
print_info(f"Opening connection to device with Serial: {_serial_to_connection}.")
handle = ljm.openS("ANY", "ANY", _serial_to_connection)  # open by serial number
print_info("Opened connection.")

info = ljm.getHandleInfo(handle)
print_info("Device info from handle:")
print_info(f"Device type: {info[0]}, Connection: {info[1]}, Serial: {info[2]}, IP: {info[3]}, Port: {info[4]}")

## Acquisition Set Points

In [ ]:
# NUM_SAMPLES = 40_000  # total points per channel
SAMPLE_RATE = 10_000  # Hz, The T8 has a maximum scan rate of 40 ksamples/second. Unlike other T-series devices, the scan rate does not depend on how many addresses are sampled per scan since all analog inputs are sampled simultaneously.
AIN_CHANNEL_NAMES = ["AIN0", "AIN1", "AIN2", "AIN4", "AIN5"]
AIN_RANGES_PER_CHANNEL = [10.0, 10.0, 3.0, 1.0, 0.010]  # set each channel to a specific range
# channels = ["AIN0"]  # use names to avoid ambiguity
SCANS_PER_READ = 256  # chunk size per eStreamRead (reasonable default)

DAC_CHANNEL_NAMES = ["DAC0", "DAC1"]  # use names to avoid ambiguity
DIO_CHANNEL_NAMES = ["DIO0", "DIO1"]  # use names to avoid ambiguity

ALL_CHANNEL_NAMES = AIN_CHANNEL_NAMES + DAC_CHANNEL_NAMES + DIO_CHANNEL_NAMES


In [ ]:
# ain_channels = {}
# ain_channels["name"] = AIN_CHANNEL_NAMES
# ain_channels["range"] = AIN_RANGES_PER_CHANNEL

# num_ch = len(ain_channels["name"])
# ain_channels

## Set the range for each AIN channel.

This is important to ensure you get the best resolution for your expected signal levels.

AIN range options for T8 (from LabJack documentation):

Ranges are: 

±11 V, ±9.6 V, ±4.8 V, ±2.4 V, ±1.2 V,

±0.6 V, ±0.3 V, ±0.15 V,

±0.075 V, ±0.036 V, ±0.018 V

In [ ]:
ain_actual_range_per_channel = []
for i, ch in enumerate(AIN_CHANNEL_NAMES):
    print_info(f"Setting range for channel {ch} to ±{AIN_RANGES_PER_CHANNEL[i]} V")
    ljm.eWriteName(handle, f"{ch}_RANGE", AIN_RANGES_PER_CHANNEL[i])

    ain_actual_range_per_channel.append(ljm.eReadName(handle, f"{ch}_RANGE"))
    print_info(f"Actual range set for {ch}: ±{ain_actual_range_per_channel[-1]:.6f} V")


## Set up acquisition rate

In [ ]:
# SAMPLE_RATE = 20000
ljm.eWriteName(handle, "AIN_SAMPLING_RATE_HZ", SAMPLE_RATE)
print_info(f"Setting up stream at {SAMPLE_RATE} Hz for ALLchannels...")

actual_sample_rate = ljm.eReadName(handle, "AIN_SAMPLING_RATE_ACTUAL_HZ")
print_info(f"Actual sample rate: {actual_sample_rate} Hz")
print_info(f"1 Sample acquisition time: {1 / actual_sample_rate * 1e3:.6f} ms")

## Read channels for inspection

In [ ]:
_res = ljm.eReadNames(handle, len(AIN_CHANNEL_NAMES), AIN_CHANNEL_NAMES)
for i, ch in enumerate(AIN_CHANNEL_NAMES):
    print_info(f"Reading back for {ch}: {_res[i]:.6f} V")

In [ ]:
for i, ch in enumerate(AIN_CHANNEL_NAMES):
    print_info(f"Reading back for {ch}: {_res[i]:.6f} V")

In [ ]:
_res = ljm.eReadNames(handle, len(ALL_CHANNEL_NAMES), ALL_CHANNEL_NAMES)
for i, ch in enumerate(AIN_CHANNEL_NAMES):
    print_info(f"Reading back for {ch}: {_res[i]:.6f} V")

## Set up DAC initial voltages

In [ ]:
print_info(f"{DAC_CHANNEL_NAMES =  }")

ljm.eWriteName(handle, "DAC0", 1.2345)
ljm.eWriteName(handle, "DAC1", 3.456)

_res = ljm.eReadNames(handle, len(DAC_CHANNEL_NAMES), DAC_CHANNEL_NAMES)
for i, ch in enumerate(DAC_CHANNEL_NAMES):
    print_info(f"Reading back for {ch}: {_res[i]:.6f} V")

## Set up DIO initial state

In [ ]:
print_info(f"{DIO_CHANNEL_NAMES =  }")

ljm.eWriteName(handle, "DIO0", 0)
ljm.eWriteName(handle, "DIO1", 0)

_res = ljm.eReadNames(handle, len(DIO_CHANNEL_NAMES), DIO_CHANNEL_NAMES)
for i, ch in enumerate(DIO_CHANNEL_NAMES):
    print_info(f"Reading back for {ch}: {_res[i]}")

# Run stream

Note that this start and stop the stream. If we dont stop, data will continuouly be saved to the buffer.

## Needed step for AIN Streaming mode

In [ ]:
# %% map names to addresses. Needed for streaming API. Also get types for reference.

# ch2stream = AIN_CHANNEL_NAMES + DAC_CHANNEL_NAMES + DIO_CHANNEL_NAMES
# NOTE: DON'T STREAM DIO. If you do, they become DI, digital inputs (which is fine if that is what you want but not what I want here since I want to toggle them as DO, digital outputs)
ch2stream = ["AIN0", "AIN1"]
_aAddresses, _aTypes = ljm.namesToAddresses(len(ch2stream), ch2stream)
# _aAddresses, _aTypes = ljm.namesToAddresses(len(DAC_CHANNEL_NAMES), DAC_CHANNEL_NAMES)
# num_ch = len(channels)


print_info(
    "Channels info:"
    + f"\n\t\t\t- ch2stream: {ch2stream}"
    + f"\n\t\t\t- aAddresses: {str(_aAddresses)}"
    + f"\n\t\t\t- aTypes: {str(_aTypes)}"
)
# reference for aTypes:https://support.labjack.com/docs/ewriteaddresses-ljm-user-s-guide#aTypes-[in]
# Values according to LJM library constants:
# 0 = UINT16, 1 = UINT32, 2 = INT32, 3 = FLOAT32, 98 = STRING, 99 = BYTE

## Settings

In [ ]:
NUM_SAMPLES = 40_000  # total points per channel
calc_acq_time = NUM_SAMPLES / SAMPLE_RATE
print_info(f"Will request {NUM_SAMPLES} scans at {SAMPLE_RATE} Hz from {len(AIN_CHANNEL_NAMES)} channels...")
print_info(f"Calculated acquisition time based on requested samples and rate: {calc_acq_time:.3f} seconds.")

## Main Stream Loop

In [ ]:
print_info(f"Requesting {NUM_SAMPLES} scans at {SAMPLE_RATE} Hz from {len(_aAddresses)} channels...")
print_info(f"Calculated acquisition time based on requested samples and rate: {calc_acq_time:.3f} seconds.")


ljm.eWriteName(handle, "DAC0", 0.0)
ljm.eWriteName(handle, "DIO1", 0)

start_t = time.perf_counter()
# ***eSTREAMSTART***
actual_rate = ljm.eStreamStart(handle, SCANS_PER_READ, len(_aAddresses), _aAddresses, SAMPLE_RATE)
print_info(f"eStream STARTED. Requested rate: {SAMPLE_RATE} Hz, Actual rate: {actual_rate:.2f} Hz")

all_flat = []
actual_scan_interval = 1.0 / actual_rate  # time between scans in seconds

_counter = 1
while True:
    # Check completion conditions
    scans_collected = len(all_flat) // len(_aAddresses)
    if scans_collected >= NUM_SAMPLES:
        break
        # note that in the streaming API, the actual acquisition continues until we stop it. Numbers are saved in the buffer and we can read them out in chunks. So we just keep reading until we have enough samples, then stop the stream. Also nte that we dont know the timestamp, only the actual_rate.

    elapsed = time.perf_counter() - start_t
    if elapsed >= calc_acq_time * 2:
        print_warning(f"Timeout reached ({elapsed:.3f}s)")
        break

    # Read and accumulate data
    # print_info(f"[INFO] Reading stream data... Scans collected so far: {scans_collected}. Time elapsed: {elapsed * 1000:.3f} ms"
    # )

    if scans_collected >= (NUM_SAMPLES // 10) * _counter:
        _counter += 1
        print(f"\nProgress: {scans_collected / NUM_SAMPLES * 100:.1f}%, ", end="", flush=True)
        print(f"Time elapsed: {time.perf_counter() - start_t:.3f} s", flush=True)

        ljm.eWriteName(handle, "DAC0", 3.3 * (1 - (_counter % 2)))
        # print_info(f"Set DAC0 to {0.1*_counter:.1f} V")

        # if _counter % 5 == 0:
        ljm.eWriteName(handle, "DIO0", 1 - (_counter % 2))
        # ljm.eWriteName(handle, "DIO1", 1-(_counter % 2))
        #     # print_info(f"Set DIO0 to {_counter % 2}")

    ret = ljm.eStreamRead(handle)

    print(".", end="", flush=True)
    if ret and ret[0]:
        all_flat.extend(ret[0])

    time.sleep(0.0001)

print("\n")
print_done(f"Time elapsed: {time.perf_counter() - start_t:.3f} s")


# ***eSTREAMSTOP***
ljm.eStreamStop(handle)
print_done("eStream STOPPED. ")

print_done(f"Total scans requested: {NUM_SAMPLES}, collected: {scans_collected}")

print_done(f"Requested rate: {SAMPLE_RATE} Hz, Actual rate: {actual_rate:.2f} Hz")

print_done(f"1 Sample acquisition time: {1 / actual_rate * 1e3:.6f} ms")

print_done(
    f"Elapsed computer time: {time.perf_counter() - start_t:.3f} s. Acquisition time: {scans_collected * actual_scan_interval:.3f} s (based on actual rate)."
)


## Convert to numpy, trim partial tail, reshape

In [ ]:
_arr_data = np.asarray(all_flat, dtype=float).reshape(scans_collected, len(_aAddresses))
# If you asked for a specific total, trim/keep only the requested scans
if _arr_data.shape[0] > NUM_SAMPLES:
    _arr_data = _arr_data[:NUM_SAMPLES, :]

# Generate timestamps for each scan
times = np.arange(_arr_data.shape[0]) * actual_scan_interval

# Combine times with data
_arr_data = np.hstack([times.reshape(-1, 1), _arr_data])

print_info(f"Collected {_arr_data.shape[0]} scans x {_arr_data.shape[1] - 1} channels (+ time column)")
print_info(f"Time range: {times[0]:.6f} to {times[-1]:.6f} seconds (session time)")


## Convert to pandas

In [ ]:
df = pd.DataFrame(_arr_data, columns=["Time"] + ch2stream)
_arr_data = None
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Check Out-of-range

In [ ]:
print_info("Checking for out-of-range values...")

for _ch in df.columns:  # skip time column
    if "AIN" not in _ch:
        continue  # only applies to AIN channels
    # print_info(f"Checking channel {_ch} for out-of-range values...")

    _range = ljm.eReadNames(handle, 1, [f"{_ch}_RANGE"])[0]
    # print(f'{_range = }')

    if df[f"{_ch}"].abs().max() > _range:
        print_warning(
            f"OUT-OF-RANGE: {ch} max absolute value {df[f'{_ch}'].abs().max():.3f} V is higher than actual channel range of ±{_range:.3f} V."
        )
    else:
        print_done(f"{_ch} is within the expected range of ±{_range:.3f} V.")


# Post Processing

In [ ]:
print_log("[POSTPROCESSING] Post Processing STARTED")


print_log("[POSTPROCESSING] Plotting data with Plotly...")
fig = go.Figure()
times = df["Time"]
for i in range(1, df.shape[1]):
    fig.add_trace(go.Scatter(x=times, y=df.iloc[:, i], mode="lines+markers", name=f"{df.columns[i]}"))

fig.update_layout(
    title="LabJack Data Acquisition",
    xaxis_title="Time (s)",
    yaxis_title="Voltage (V)",
    hovermode="x unified",
    template="plotly",
)
# Save HTML and open in VS Code webview
html_file = f"Results\\{datenow_str()}_labjack_acquisition_plot.html"

if save_plot:
    fig.write_html(html_file)
    print_log(f"[POSTPROCESSING] Plot saved to .\\{html_file}\n")

fig.show()

# END - Close connections

In [ ]:
if close_connections_at_end:
    try:
        ljm.close(handle)
        print_info("Closed connection to hardware.")
    except Exception as e:
        print_error(f"Close error: {e}")

    print_info("### Acquisition complete ###\n")

    print_info(f"Total scripts execution time: {time.time() - start_t:.2f} seconds\n")